# FriskvårdsCentrum Nordic – ETL Pipeline & Analysis

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
from transformers import pipeline

In [ ]:
df = pd.read_csv('friskvard_data.csv')
df = df.drop_duplicates()
df.head()

In [ ]:
df['passdatum'] = pd.to_datetime(df['passdatum'], errors='coerce', dayfirst=True)
df['bokningsdatum'] = pd.to_datetime(df['bokningsdatum'], errors='coerce', dayfirst=True)

In [ ]:
df['weekday'] = df['passdatum'].dt.weekday
df['month'] = df['passdatum'].dt.month
df['age'] = 2024 - df['födelseår']

In [ ]:
sentiment_model = pipeline('sentiment-analysis')

def analyze_sentiment(text):
    if pd.isna(text):
        return None
    result = sentiment_model(str(text)[:512])[0]
    return result['label'].lower()

df['sentiment'] = df['feedback_text'].apply(analyze_sentiment)

In [ ]:
cancel_rate = df.groupby('month')['status'].apply(lambda x: (x=='cancelled').mean())
cancel_rate.plot(kind='bar')
plt.title('Avbokningsfrekvens per månad')
plt.show()

In [ ]:
bookings = df.groupby('weekday').size()
bookings.plot(kind='bar')
plt.title('Bokningar per veckodag')
plt.show()

In [ ]:
conn = sqlite3.connect('friskvard.db')
df.to_sql('bookings', conn, if_exists='replace', index=False)
conn.close()